# 04 - Methode 2 : variables transformees par ACP

IFT599-1 / IFT799-51 / IFT799-52 - TP1

Membres : Nom Prenom - CIP

Objectif : generer 15 composantes principales et y appliquer la meme
selection de paires que sur les variables originales.

Source : scikit-learn, sklearn.decomposition.PCA et sklearn.preprocessing.StandardScaler.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src.config import CLASSES, N_COMPOSANTES, SEED
from src.loading import charger
from src.selection import selectionner
from src.plotting import nuage

X, y = charger()

## ACP

Les variables sont standardisees avant l'ACP (diapositives 41 et 42) :
les variances des genes different fortement.

In [ ]:
Z = StandardScaler().fit_transform(X)

acp = PCA(n_components=N_COMPOSANTES, random_state=SEED)
C = acp.fit_transform(Z)

print("variance expliquee cumulee :", acp.explained_variance_ratio_.sum().round(3))
acp.explained_variance_ratio_.round(3)

## Meilleure paire de composantes pour toutes les paires de classes

In [ ]:
paire_globale, meilleures = selectionner(C, y, CLASSES)
print("paire globale :", paire_globale)

nuage(C, y, CLASSES, paire_globale,
      "ACP - meilleure paire globale",
      "../figures/pca/globale.png")

## Meilleure paire de composantes pour chaque paire de classes

In [ ]:
lignes = []
for (a, b), (cols, o) in sorted(meilleures.items()):
    lignes.append({"paire": f"{a}-{b}", "composantes": cols, "overlap": round(o, 3)})

    nuage(C, y, [a, b], cols,
          f"ACP - {a} vs {b}",
          f"../figures/pca/{a}_{b}.png")

t = pd.DataFrame(lignes)
t["bien_separees"] = t["overlap"] < 1
t.to_csv("../results/selection_acp.csv", index=False)
t

### Commentaires

Une seule paire passe sous le seuil (COAD-PRAD, 0.682), contre cinq avec les
genes originaux. L'ACP separe donc moins bien.

C'est la mise en garde de la diapositive 52 : une forte variance n'est pas
synonyme de pertinence. L'ACP maximise la variance, pas la separation entre
classes, et elle ignore completement les etiquettes.

Les 15 composantes ne retiennent que 52.7 % de la variance, sans coude marque :
l'information est repartie sur de nombreuses directions.

Les premieres composantes dominent la selection (CP0 pour quatre paires, CP1
pour trois), ce qui suggere que l'information discriminante s'y trouve, mais
melangee a une variabilite non pertinente.